# Hypothesis Testing and Reliability Analysis

## Introduction

This notebook moves from exploratory analysis to formal statistical testing. The goal is to evaluate whether the strongest patterns identified in Notebook 02 remain supported when compared across failure-related or degradation-related groups. The MetroPT-3 and hydraulic datasets are analyzed separately first, because they differ fundamentally in structure and labeling, and are then interpreted together in relation to the project's three hypotheses on within-dataset sensor change, cross-dataset similarity, and reliability-oriented context.

### Notebook structure

This notebook is organized into the following sections:

- **Setup** - load datasets, libraries, and project-specific configuration
- **Hypotheses and statistical approach** - the three hypotheses and overall statistical strategy
- **Statistical methods** - formal definitions of the tests used in this notebook
- **MetroPT-3 testing** - pre-failure versus normal-operation comparison
- **Hydraulic testing** - condition-based comparisons across labeled groups
- **Reliability metrics** - MTBF and MTTR analysis with cross-dataset context
- **Cross-dataset interpretation** - synthesis of findings across both systems
- **Conclusions** - hypothesis-by-hypothesis verdict and methodological note

## Setup

This section loads the processed datasets prepared in Notebook 01, the documented MetroPT-3 failure events, the statistical libraries used for hypothesis testing, and the variable selections carried forward from Notebook 02. The display settings are configured to render full content in result tables.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, kruskal
from statsmodels.stats.multitest import multipletests

# Reproducibility and display
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option('display.max_colwidth', None)

# Load processed datasets
metro = pd.read_csv('../data/processed/metro_cleaned.csv', parse_dates=['timestamp'])
hydraulics = pd.read_csv('../data/processed/hydraulics_features.csv')

# Load MetroPT-3 documented failure events
metro_failures = pd.read_csv('../data/raw/metropt3_failure_reports.csv', parse_dates=['start_time', 'end_time'], keep_default_na=False)
metro_failures['duration'] = metro_failures['end_time'] - metro_failures['start_time']

print(f"MetroPT-3 shape: {metro.shape}")
print(f"Hydraulic shape: {hydraulics.shape}")
print(f"\nMetroPT-3 failure events: {len(metro_failures)}")
print(metro_failures)

# Variable selections guided by Notebook 02
metro_vars = ['TP2', 'H1', 'DV_pressure', 'Oil_temperature', 'Motor_current']

hydraulic_pairs = {
    'cooler_condition': ['TS1_mean', 'TS1_std'],
    'pump_leakage': ['EPS1_mean', 'PS1_mean'],
    'stable_flag': ['TS1_std', 'VS1_std', 'PS1_std', 'EPS1_std'],
}

MetroPT-3 shape: (1516948, 16)
Hydraulic shape: (2205, 73)

MetroPT-3 failure events: 4
   failure_id          start_time            end_time failure_type  \
0           1 2020-04-18 00:00:00 2020-04-18 23:59:00     Air leak   
1           2 2020-05-29 23:30:00 2020-05-30 06:00:00     Air leak   
2           3 2020-06-05 10:00:00 2020-06-07 14:30:00     Air leak   
3           4 2020-07-15 14:30:00 2020-07-15 19:00:00     Air leak   

      severity                    report_note        duration  
0  High stress                                0 days 23:59:00  
1  High stress  Maintenance on 30Apr at 12:00 0 days 06:30:00  
2  High stress   Maintenance on 8Jun at 16:00 2 days 04:30:00  
3  High stress  Maintenance on 16Jul at 00:00 0 days 04:30:00  


---

## Hypotheses and statistical approach

### Hypothesis 1

Selected sensor variables differ measurably between normal and failure-related conditions within each dataset when the datasets are analyzed separately.

In MetroPT-3, this hypothesis refers to differences between normal operating periods and pre-failure time windows preceding the documented failure events. In the hydraulic dataset, it refers to differences between feature values across labeled degradation states or between stable and unstable cycles.

### Hypothesis 2

The MetroPT-3 metro air compressor and the hydraulic test rig show analogous failure-related behavior in their shared physical sensing concepts - pressure, temperature, and electrical load-related variables - even though the two systems differ in design, monitoring structure, and operating context.

Because failure is represented differently in the two datasets, this hypothesis is tested by conceptual analogy rather than by direct one-to-one equivalence. In MetroPT-3, failure-related behavior is inferred from time windows preceding documented incidents. In the hydraulic dataset, it is inferred from condition labels and cycle-level degradation structure.

### Hypothesis 3

Reliability-oriented measures - Mean Time Between Failures (MTBF) and Mean Time To Repair (MTTR) - provide contextual support for the interpretation that sensor behavior does not change randomly, but is associated with meaningful periods of degradation, instability, or failure-related transition. These metrics are not treated as primary evidence for the other two hypotheses, but as supporting context for the timing and operational significance of detected sensor patterns.

### Statistical approach

The hypothesis tests use nonparametric methods throughout, because the sensor distributions across operating regimes vary substantially in shape and are not assumed to be normal. Two-group comparisons use the Mann-Whitney U test; multi-level comparisons use the Kruskal-Wallis omnibus test, optionally followed by pairwise Mann-Whitney comparisons when the omnibus result is significant. The family-wise error rate across primary tests is controlled by the Bonferroni correction.

Following the project's planned interpretation rule, support for each hypothesis is evaluated through the combination of statistical significance, the direction and magnitude of the effect, the consistency of the result with the physical interpretation of the sensors, and the convergence of evidence across the two datasets. Significance alone is treated as one criterion among several rather than as the decisive measure.

---

## Statistical methods

This section defines the formal tests applied in the hypothesis testing that follows. The Mann-Whitney U test is used for two-group comparisons in both datasets. The Kruskal-Wallis omnibus test is used for multi-level comparisons in the hydraulic dataset, with pairwise Mann-Whitney comparisons applied as a follow-up step when the omnibus result is significant. Effect sizes are reported alongside p-values using the rank-biserial correlation. The family-wise error rate across primary tests is controlled by the Bonferroni correction.

### Mann-Whitney U test

The Mann-Whitney U test is a non-parametric test of whether two
independent samples are drawn from the same distribution. Given samples $X = \{x_1, \ldots, x_{n_1}\}$ and $Y = \{y_1, \ldots, y_{n_2}\}$, all $n_1 + n_2$ observations are pooled and ranked. The test statistic is:

$$U_1 = \sum_{i=1}^{n_1} \sum_{j=1}^{n_2} S(x_i, y_j)$$

where $S(x_i, y_j) = 1$ if $x_i > y_j$, $0.5$ if $x_i = y_j$, and $0$ otherwise. The complementary statistic $U_2 = n_1 n_2 - U_1$, and the reported test statistic is $U = \min(U_1, U_2 $.

The null hypothesis $H_0$ states that $X$ and $Y$ are drawn from the same distribution. Under $H_0$, $U$ follows a known distribution from which the two-sided p-value is computed.

The Mann-Whitney U test doesn't assume normality and is robust to outliers, which makes it appropriate for sensor data where the distributional shape varies across operating regimes.

### Effect size: rank-biserial correlation

Statistical significance alone doesn't indicate the size of a difference. The rank-biserial correlation $r$ is the natural effect size for Mann-Whitney U:

$$r = 1 - \frac{2U}{n_1 n_2}$$

where $U$ is the test statistic and $n_1, n_2$ are the group sizes. The value $r$ ranges from $-1$ to $+1$, where $0$ indicates no difference between groups, positive values indicate that the first group's values tend to exceed the second's, and negative values indicate the reverse.

Common interpretation thresholds: $|r| < 0.1$ negligible, $|r| < 0.3$ small, $|r| < 0.5$ medium, $|r| \geq 0.5$ large. These thresholds are used as a reference for evaluating the practical significance of the test results in this notebook.

### Kruskal-Wallis omnibus test

The Kruskal-Wallis test is a non-parametric extension of Mann-Whitney U to more than two independent groups. Given $k$ groups with sample sizes $n_1, n_2, \ldots, n_k$ and total sample size $N = \sum_{i=1}^{k} n_i$, all observations are pooled and ranked. The test statistic is:

$$H = \frac{12}{N(N+1)} \sum_{i=1}^{k} \frac{R_i^2}{n_i} - 3(N+1)$$

where $R_i$ is the sum of ranks for group $i$.

The null hypothesis $H_0$ states that all $k$ groups are drawn from the same distribution. Under $H_0$, $H$ approximately follows a chi-squared distribution with $k - 1$ degrees of freedom, from which the p-value is computed.

Like the Mann-Whitney U test, the Kruskal-Wallis test doesn't assume normality. It is appropriate for the hydraulic-dataset feature distributions that vary across operating regimes and degradation states. When a Kruskal-Wallis result is significant, pairwise Mann-Whitney comparisons can be applied as a next step to identify which specific group pairs differ.

### Multiple-comparison correction

When $k$ hypothesis tests are conducted simultaneously at significance level $\alpha$, the family-wise error rate increases above $\alpha$. The Bonferroni correction adjusts the per-test threshold to:

$$\alpha_{\text{corrected}} = \frac{\alpha}{k}$$

Equivalently, raw p-values can be compared against the original $\alpha$ threshold after multiplying by $k$, with values exceeding 1 clamped at 1. For the $k = 5$ primary hydraulic tests planned in this notebook, the corrected threshold is $\alpha_{\text{corrected}} = 0.010$.

The Bonferroni correction is conservative - it controls the family-wise error rate strictly but reduces statistical power. It is appropriate when the cost of false positives is high relative to false negatives, which is the standard expectation in confirmatory analysis.

---

## MetroPT-3: pre-failure versus normal-operation comparison

### Failure-event definition

The MetroPT-3 dataset provides four documented failure intervals, all classified as air leaks under high stress. Pre-failure windows are anchored to each failure start time, and matched normal-operation windows are extracted from two days prior with a 48-hour buffer. The failure intervals themselves are excluded from both groups.

A minimum coverage threshold of $90\%$ is applied to each window independently. Failure 4 is excluded because its matched normal window covers only $4.48$ hours of the requested 6, falling below the threshold. Three failure events are retained for comparison.

In [2]:
# MetroPT-3 window extraction helpers

def extract_prefailure_windows(df, failure_df, hours):
    windows = []

    for _, row in failure_df.iterrows():
        ft = row['start_time']
        start = ft - pd.Timedelta(hours=hours)

        mask = (df['timestamp'] >= start) & (df['timestamp'] < ft)
        part = df.loc[mask].copy()
        part['failure_id'] = row['failure_id']
        part['failure_start'] = ft
        windows.append(part)

    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()


def extract_normal_windows(df, failure_df, hours, buffer_hours=48):
    windows = []

    for _, row in failure_df.iterrows():
        ft = row['start_time']
        start = ft - pd.Timedelta(hours=hours + buffer_hours)
        end = ft - pd.Timedelta(hours=buffer_hours)

        mask = (df['timestamp'] >= start) & (df['timestamp'] < end)
        part = df.loc[mask].copy()
        part['failure_id'] = row['failure_id']
        part['failure_start'] = ft
        windows.append(part)

    return pd.concat(windows, ignore_index=True) if windows else pd.DataFrame()

In [3]:
prefail_6h = extract_prefailure_windows(metro, metro_failures, 6)
normal_6h = extract_normal_windows(metro, metro_failures, 6, buffer_hours=48)

In [4]:
def filter_windows_by_coverage(df, requested_hours, min_fraction=0.9):
    coverage = (
        df.groupby('failure_id')['timestamp']
        .agg(actual_start='min', actual_end='max')
        .reset_index()
    )

    coverage['coverage_hours'] = (
        coverage['actual_end'] - coverage['actual_start']
    ).dt.total_seconds() / 3600

    coverage['keep'] = coverage['coverage_hours'] >= requested_hours * min_fraction

    keep_ids = coverage.loc[coverage['keep'], 'failure_id']
    filtered = df[df['failure_id'].isin(keep_ids)].copy()

    return filtered, coverage

In [5]:
# Apply coverage filter independently to pre-failure and normal windows
prefail_6h_filtered, prefail_coverage = filter_windows_by_coverage(
    prefail_6h, requested_hours=6
)
normal_6h_filtered, normal_coverage = filter_windows_by_coverage(
    normal_6h, requested_hours=6
)

# Keep only failures that survived both filters (pairing rule)
kept_ids = set(prefail_coverage.loc[prefail_coverage['keep'], 'failure_id']) & \
           set(normal_coverage.loc[normal_coverage['keep'], 'failure_id'])

prefail_6h_clean = prefail_6h_filtered[prefail_6h_filtered['failure_id'].isin(kept_ids)].copy()
normal_6h_clean  = normal_6h_filtered[normal_6h_filtered['failure_id'].isin(kept_ids)].copy()

print("Pre-failure coverage:")
print(prefail_coverage)

print("\nNormal coverage:")
print(normal_coverage)

print(f"\nFailure events surviving the coverage rule: {sorted(kept_ids)}")
print(f"Pre-failure rows: {len(prefail_6h_clean):,}")
print(f"Normal rows:      {len(normal_6h_clean):,}")

Pre-failure coverage:
   failure_id        actual_start          actual_end  coverage_hours  keep
0           1 2020-04-17 18:00:10 2020-04-17 23:59:49        5.994167  True
1           2 2020-05-29 17:30:08 2020-05-29 23:29:58        5.997222  True
2           3 2020-06-05 04:00:06 2020-06-05 09:59:54        5.996667  True
3           4 2020-07-15 08:30:00 2020-07-15 14:29:51        5.997500  True

Normal coverage:
   failure_id        actual_start          actual_end  coverage_hours   keep
0           1 2020-04-15 18:00:09 2020-04-15 23:59:59        5.997222   True
1           2 2020-05-27 17:30:09 2020-05-27 23:29:55        5.996111   True
2           3 2020-06-03 04:00:03 2020-06-03 09:59:54        5.997500   True
3           4 2020-07-13 08:30:06 2020-07-13 12:59:05        4.483056  False

Failure events surviving the coverage rule: [1, 2, 3]
Pre-failure rows: 6,148
Normal rows:      6,144


#### Observations

The coverage filter retains three of the four documented failure events. Failure 4 is excluded because its matched normal window covers only $4.48$ of the requested $6$ hours, falling below the $90\%$ threshold. For the retained events, the pre-failure and normal windows are closely matched in size ($6{,}148$ vs $6{,}144$ rows).

### Window choice

Four candidate pre-failure window lengths are evaluated: $1$, $6$, $12$, and $24$ hours. For each length, the coverage filter is applied, per-event medians are computed across the retained failures, and the direction of the pre-failure shift is summarized for each variable.

In [6]:
# Window-length sweep: retain matched failures and summarize pre-failure direction

WINDOW_LENGTHS_HOURS = [1, 6, 12, 24]
COVERAGE_THRESHOLD = 0.9

sweep_records = []
window_retention_records = []

for h in WINDOW_LENGTHS_HOURS:
    # Extract windows
    prefail = extract_prefailure_windows(metro, metro_failures, h)
    normal = extract_normal_windows(metro, metro_failures, h, buffer_hours=48)

    # Apply coverage filter independently, then enforce matched failures
    prefail_filt, prefail_cov = filter_windows_by_coverage(
        prefail, requested_hours=h, min_fraction=COVERAGE_THRESHOLD
    )
    normal_filt, normal_cov = filter_windows_by_coverage(
        normal, requested_hours=h, min_fraction=COVERAGE_THRESHOLD
    )

    kept_ids = sorted(
        set(prefail_cov.loc[prefail_cov['keep'], 'failure_id']) &
        set(normal_cov.loc[normal_cov['keep'], 'failure_id'])
    )

    window_retention_records.append({
        'window_h': h,
        'retained_failures': str(kept_ids),
        'n_retained': len(kept_ids)
    })

    prefail_clean = prefail_filt[prefail_filt['failure_id'].isin(kept_ids)].copy()
    normal_clean = normal_filt[normal_filt['failure_id'].isin(kept_ids)].copy()

    # Per-event medians
    for fid in kept_ids:
        for var in metro_vars:
            pre_median = prefail_clean.loc[
                prefail_clean['failure_id'] == fid, var
            ].median()
            norm_median = normal_clean.loc[
                normal_clean['failure_id'] == fid, var
            ].median()

            sweep_records.append({
                'window_h': h,
                'failure_id': fid,
                'variable': var,
                'prefail_minus_normal': pre_median - norm_median
            })

sweep_df = pd.DataFrame(sweep_records)
window_retention_df = pd.DataFrame(window_retention_records)

window_retention_df

,window_h,retained_failures,n_retained
0,1,"[1, 2, 3]",3
1,6,"[1, 2, 3]",3
2,12,"[1, 2, 3]",3
3,24,"[1, 2, 3]",3


In [7]:
# Direction summary by variable and window length

consistency_summary = (
    sweep_df.groupby(['window_h', 'variable'])['prefail_minus_normal']
    .agg(
        median_diff='median',
        n_negative=lambda x: (x < 0).sum(),
        n_positive=lambda x: (x > 0).sum(),
        n_zero=lambda x: (x == 0).sum()
    )
    .reset_index()
)

consistency_summary['direction_summary'] = (
    consistency_summary['n_negative'].astype(str) + 'n / ' +
    consistency_summary['n_positive'].astype(str) + 'p / ' +
    consistency_summary['n_zero'].astype(str) + 'z'
)

direction_pivot = consistency_summary.pivot(
    index='variable',
    columns='window_h',
    values='direction_summary'
)

direction_pivot

window_h,1,6,12,24
variable,,,,
DV_pressure,1n / 0p / 2z,1n / 0p / 2z,1n / 0p / 2z,0n / 0p / 3z
H1,2n / 1p / 0z,2n / 1p / 0z,2n / 1p / 0z,1n / 2p / 0z
Motor_current,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z
Oil_temperature,2n / 1p / 0z,3n / 0p / 0z,2n / 1p / 0z,1n / 2p / 0z
TP2,2n / 0p / 1z,2n / 0p / 1z,1n / 0p / 2z,1n / 0p / 2z


In [8]:
consistency_summary = (
    sweep_df.groupby(['window_h', 'variable'])['prefail_minus_normal']
    .agg(
        median_diff='median',
        n_negative=lambda x: (x < 0).sum(),
        n_positive=lambda x: (x > 0).sum(),
        n_zero=lambda x: (x == 0).sum()
    )
    .reset_index()
)
# Sorting
consistency_summary.sort_values(['variable', 'window_h'])
consistency_summary

,window_h,variable,median_diff,n_negative,n_positive,n_zero
0,1,DV_pressure,0.0000,1,0,2
1,1,H1,-0.4580,2,1,0
2,1,Motor_current,0.0025,1,2,0
3,1,Oil_temperature,-0.9500,2,1,0
4,1,TP2,-0.0020,2,0,1
5,6,DV_pressure,0.0000,1,0,2
6,6,H1,-0.0640,2,1,0
7,6,Motor_current,0.0025,1,2,0
8,6,Oil_temperature,-0.6250,3,0,0
9,6,TP2,-0.0020,2,0,1


In [9]:
# Consistency count

# Direction summary string per (variable, window_h)
consistency_summary['direction_summary'] = (
    consistency_summary['n_negative'].astype(str) + 'n / ' +
    consistency_summary['n_positive'].astype(str) + 'p / ' +
    consistency_summary['n_zero'].astype(str) + 'z'
)

direction_pivot = consistency_summary.pivot(
    index='variable',
    columns='window_h',
    values='direction_summary'
)
direction_pivot

window_h,1,6,12,24
variable,,,,
DV_pressure,1n / 0p / 2z,1n / 0p / 2z,1n / 0p / 2z,0n / 0p / 3z
H1,2n / 1p / 0z,2n / 1p / 0z,2n / 1p / 0z,1n / 2p / 0z
Motor_current,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z,1n / 2p / 0z
Oil_temperature,2n / 1p / 0z,3n / 0p / 0z,2n / 1p / 0z,1n / 2p / 0z
TP2,2n / 0p / 1z,2n / 0p / 1z,1n / 0p / 2z,1n / 0p / 2z


#### Observations

The window-length sweep suggests that the 6-hour pre-failure window provides the clearest and most interpretable MetroPT-3 signal. `Oil_temperature` shows the strongest consistency at this window length, with all three retained failures exhibiting lower pre-failure medians than their matched normal windows. `H1` shows a weaker but still partly consistent downward shift across the shorter windows.

By contrast, `Motor_current` shows mixed direction across events at every tested window length, indicating that its variation reflects operating-regime changes rather than a pre-failure signature. `DV_pressure` shows almost no systematic change between pre-failure and normal windows at any window length. `TP2` shows a partial signal - two of three failures with lower pre-failure medians, the third flat - placing it as a weaker but coherent secondary candidate.

Based on this sweep, the 6-hour window is treated as the main MetroPT-3 comparison window, while 12 hours may be retained as a sensitivity check if needed.

### Event-level summary logic

The rows within each extracted window are temporally dependent - treating them as independent observations would inflate the effective sample size and invalidate the test assumptions. To address this, each window is summarized at the event level by computing the per-failure median for each selected variable. This reduces the comparison to $n = 3$ vs $n = 3$ matched event summaries.

In [10]:
# Summarize each event-window

def summarize_windows(df, variables, group_label):
    summary = (
        df.groupby('failure_id')[variables]
        .median()
        .reset_index()
    )
    summary['group'] = group_label
    return summary

In [11]:
prefail_6h_summary = summarize_windows(prefail_6h_clean, metro_vars, 'prefail_6h')
normal_6h_summary = summarize_windows(normal_6h_clean, metro_vars, 'normal_6h')

metro_event_summary_6h = pd.concat(
    [prefail_6h_summary, normal_6h_summary],
    ignore_index=True
)

metro_event_summary_6h

,failure_id,TP2,H1,DV_pressure,Oil_temperature,Motor_current,group
0,1,-0.018,8.238,-0.024,49.450,0.0400,prefail_6h
1,2,-0.012,8.764,-0.020,63.975,0.0425,prefail_6h
2,3,-0.012,8.800,-0.022,63.475,0.0450,prefail_6h
3,1,-0.014,8.886,-0.024,57.725,0.0425,normal_6h
4,2,-0.010,8.158,-0.018,64.600,0.0400,normal_6h
5,3,-0.012,8.864,-0.022,63.775,0.0425,normal_6h


#### Observations

After summarizing the retained 6-hour windows at the failure-event level, the clearest and most consistent pre-failure shift appears in `Oil_temperature`: all three retained failures show lower median oil temperature in the pre-failure window than in the matched normal window.

`H1` shows a weaker but still partly coherent pattern, with lower pre-failure medians in two of the three retained failures. `TP2` also shows a partial signal, with a pre-failure decrease in two failures and no change in the third. By contrast, `DV_pressure` shows almost no systematic difference, and `Motor_current` changes in inconsistent directions across events.

At the 6-hour window level, the most promising MetroPT-3 candidates for formal comparison therefore appear to be `Oil_temperature` as the strongest signal, with `H1` and `TP2` retained as weaker supporting candidates.

### Formal test results

Three variables are carried forward to formal testing: `Oil_temperature`, `TP2`, and `H1`. `Motor_current` and `DV_pressure` are excluded based on the window-length sweep findings reported above.

The tests are conducted at the event-summary level using the Mann-Whitney U test, with the per-failure median as the unit of comparison rather than individual sensor rows. This protects against the within-window autocorrelation that would inflate row-level significance, but reduces the sample size to $n = 3$ vs $n = 3$ after the coverage filter.

The rank-biserial correlation is reported alongside each result as the effect size measure. Bonferroni correction across $k = 3$ tests sets the corrected threshold at $\alpha = 0.05 / 3 \approx 0.0167$.

In [12]:
def mannwhitney_table(df_a, df_b, variables, group_a='A', group_b='B'):
    """Mann-Whitney U for each variable, with Bonferroni-corrected
    p-values, rank-biserial effect size, sample sizes, and direction."""
    results = []
    for var in variables:
        x = df_a[var].dropna()
        y = df_b[var].dropna()

        stat, p = mannwhitneyu(x, y, alternative='two-sided')

        n1, n2 = len(x), len(y)
        rank_biserial = 1 - (2 * stat) / (n1 * n2)

        diff = x.median() - y.median()
        if diff > 0:
            direction = f'{group_a} > {group_b}'
        elif diff < 0:
            direction = f'{group_b} > {group_a}'
        else:
            direction = 'equal'

        results.append({
            'variable':           var,
            f'n_{group_a}':       n1,
            f'n_{group_b}':       n2,
            f'{group_a}_median':  x.median(),
            f'{group_b}_median':  y.median(),
            'direction':          direction,
            'u_stat':             stat,
            'p_value':            p,
            'rank_biserial':      rank_biserial,
        })

    out = pd.DataFrame(results)
    out['p_bonf'] = multipletests(out['p_value'], method='bonferroni')[1]
    return out.sort_values('p_value')

In [13]:
metro_vars_final_6h = ['Oil_temperature', 'TP2', 'H1']

In [14]:
metro_event_results_6h = mannwhitney_table(
    prefail_6h_summary,
    normal_6h_summary,
    metro_vars_final_6h,
    group_a='prefail',
    group_b='normal'
)

metro_event_results_6h

,variable,n_prefail,n_normal,prefail_median,normal_median,direction,u_stat,p_value,rank_biserial,p_bonf
1,TP2,3,3,-0.012,-0.012,equal,3.0,0.642835,0.333333,1.0
0,Oil_temperature,3,3,63.475,63.775,normal > prefail,3.0,0.700000,0.333333,1.0
2,H1,3,3,8.764,8.864,normal > prefail,3.0,0.700000,0.333333,1.0


#### Observations

The event-level Mann-Whitney U results for the retained 6-hour MetroPT-3 windows do not reach statistical significance after Bonferroni correction. This is expected, given that only three matched failure events remain available after the coverage filter, which makes formal inference very low-powered.

Even so, the direction of the event-level medians remains informative. `Oil_temperature` shows lower pre-failure medians than the matched normal windows in all three retained failures, consistent with the earlier descriptive inspection. `H1` shows a weaker but still partly coherent pattern, with lower pre-failure medians in two of the three retained failures. By contrast, `TP2` does not show a clear grouped difference at the event-summary level, even though the per-event view showed lower pre-failure medians in two of the three retained failures. This illustrates that the median-of-three event summary is a stricter statistic than simple direction counting and may hide partial shifts that are not consistent across most events.

### MetroPT-3 interpretation

The MetroPT-3 formal results are limited by the small number of matched failure events available after the coverage filter. With $n = 3$ vs $n = 3$ at the event level, the tests have very low statistical power, and non-significant results are expected even when a real directional pattern may be present.

The directional evidence remains informative nonetheless. `Oil_temperature` shows the clearest and most consistent pre-failure pattern across the retained events, supporting the physical interpretation that thermal behavior changes before documented failures. `H1` provides weaker supporting evidence, while `TP2` remains a partial but less stable signal at the grouped level.

The MetroPT-3 evidence is therefore treated as directional and exploratory-supportive rather than as strong standalone statistical confirmation.

---

## Hydraulic system: condition-based formal testing

### Label structure and test design

The hydraulic dataset is cycle-based and includes explicit condition labels for `cooler_condition`, `valve_condition`, `pump_leakage`, `accumulator_pressure`, and `stable_flag`. Unlike MetroPT-3, no temporal window construction is needed - each row of the cycle-aggregated feature table corresponds to a single labeled cycle, and groups are defined directly by label values.

The exploratory analysis in Notebook 02 identified the strongest feature-condition relationships as `TS1_mean` and `TS1_std` for `cooler_condition`, `EPS1_mean` for `pump_leakage`, and standard-deviation features for `stable_flag`. `valve_condition` and `accumulator_pressure` showed weaker EDA separation and are excluded from formal testing.

Multi-level labels are tested with the Kruskal-Wallis omnibus test, followed by pairwise Mann-Whitney comparisons when the omnibus result is significant. Binary labels are tested directly with Mann-Whitney U. The total formal-test count is five, setting the Bonferroni-corrected threshold at $\alpha = 0.05 / 5 = 0.010$.

In [15]:
# Compact summary of hydraulic label structure

hydraulic_label_cols = [
    'cooler_condition',
    'valve_condition',
    'pump_leakage',
    'accumulator_pressure',
    'stable_flag'
]

label_summary_records = []

for col in hydraulic_label_cols:
    counts = hydraulics[col].value_counts().sort_index()
    shares = hydraulics[col].value_counts(normalize=True).sort_index()

    for level in counts.index:
        label_summary_records.append({
            'label': col,
            'level': level,
            'n_cycles': counts[level],
            'share': round(shares[level], 4)
        })

hydraulic_label_summary = pd.DataFrame(label_summary_records)


hydraulic_label_summary

,label,level,n_cycles,share
0,cooler_condition,3,732,0.3320
1,cooler_condition,20,732,0.3320
2,cooler_condition,100,741,0.3361
3,valve_condition,73,360,0.1633
4,valve_condition,80,360,0.1633
5,valve_condition,90,360,0.1633
6,valve_condition,100,1125,0.5102
7,pump_leakage,0,1221,0.5537
8,pump_leakage,1,492,0.2231
9,pump_leakage,2,492,0.2231


#### Observations

Each label has a distinct structure. `cooler_condition` is a three-level variable with near-perfect group balance (732, 732, 741 cycles), well-suited to a Kruskal-Wallis omnibus test with possible follow-up pairwise comparisons. `pump_leakage` is a three-level variable dominated by the healthy class (1221 cycles vs 492 each for weak and severe leakage); the cleanest test is an endpoint contrast between full health (0) and severe leakage (2). `stable_flag` is already binary (1449 stable, 756 unstable) and tests directly with Mann-Whitney U.

`valve_condition` and `accumulator_pressure` show four-level imbalanced distributions and were already flagged as weaker EDA candidates in Notebook 02. Both are excluded from formal testing and reported descriptively only.

The total formal-test count is five: two features for `cooler_condition`, one for `pump_leakage`, and two for `stable_flag`. The Bonferroni-corrected significance threshold for primary tests is therefore $\alpha = 0.05 / 5 = 0.010$. Optional features within the same physical concept are run as exploratory checks and are not included in the Bonferroni count.

### Cooler condition

The `cooler_condition` label has three levels - $3$ (close-to-total failure), $20$ (reduced efficiency), and $100$ (full efficiency) - with near-perfect group balance ($732$, $732$, $741$ cycles). The Kruskal-Wallis omnibus test is applied to the two strongest features identified in Notebook 02: `TS1_mean` and `TS1_std`. Pairwise Mann-Whitney comparisons are run as a next step because the omnibus result is significant after correction.

In [16]:
# Kruskal-Wallis on TS1_mean and TS1_std by cooler_condition
# Global Bonferroni correction across k = 5 primary hydraulic tests

GLOBAL_K = 5
GLOBAL_ALPHA = 0.05 / GLOBAL_K  # 0.010

cooler_features = ['TS1_mean', 'TS1_std']
cooler_levels = sorted(hydraulics['cooler_condition'].unique())

cooler_kw_results = []

for var in cooler_features:
    groups = [
        hydraulics.loc[hydraulics['cooler_condition'] == lvl, var].dropna()
        for lvl in cooler_levels
    ]
    stat, p = kruskal(*groups)

    cooler_kw_results.append({
        'variable': var,
        'levels': cooler_levels,
        'group_sizes': [len(g) for g in groups],
        'group_medians': [g.median() for g in groups],
        'H_statistic': stat,
        'p_value': p,
        'alpha_bonf': GLOBAL_ALPHA,
        'significant': p < GLOBAL_ALPHA,
    })

cooler_kw_df = pd.DataFrame(cooler_kw_results)
# Clean display formatting
cooler_kw_df['group_medians'] = cooler_kw_df['group_medians'].apply(
    lambda vals: [round(v, 4) for v in vals]
)
cooler_kw_df['H_statistic'] = cooler_kw_df['H_statistic'].round(4)
cooler_kw_df['p_value'] = cooler_kw_df['p_value'].apply(lambda p: f"{p:.3e}")

cooler_kw_df

,variable,levels,group_sizes,group_medians,H_statistic,p_value,alpha_bonf,significant
0,TS1_mean,"[3, 20, 100]","[732, 732, 741]","[55.6925, 44.9063, 35.8881]",1915.5335,0.000e+00,0.01,True
1,TS1_std,"[3, 20, 100]","[732, 732, 741]","[0.2279, 0.1961, 0.1507]",1703.3084,0.000e+00,0.01,True


#### Observations

The Kruskal-Wallis omnibus test produces highly significant results for both `TS1_mean` and `TS1_std` (`H = 1915.5` and `H = 1703.3`, respectively, with both p-values effectively zero). Both results are far below the Bonferroni-corrected threshold of $\alpha = 0.010$, providing strong support for the hydraulic part of Hypothesis 1 in the case of `cooler_condition`.

The group medians show clear monotonic separation across cooler-efficiency levels. For `TS1_mean`, the median temperature decreases from 55.7°C (3% efficiency) through 44.9°C (20% efficiency) to 35.9°C (full efficiency), consistent with the physical expectation that a degraded cooler fails to dissipate heat effectively. For `TS1_std`, the within-cycle variability also decreases monotonically, from approximately 0.228 to 0.196 and then to 0.151, indicating that poorer cooler condition is associated not only with higher temperature but also with less stable thermal behavior during each cycle.

These findings are the strongest formal results obtained in the project. They confirm the cooler-condition pattern identified descriptively in Notebook 02 and show that engineered per-cycle features can separate degradation states very clearly for at least one hydraulic component.

#### Pairwise comparisons

Because the omnibus Kruskal-Wallis result is significant for both cooler-related features, pairwise Mann-Whitney comparisons are run to confirm whether the separation holds across each pair of condition levels. Bonferroni correction within the pairwise family covers $2$ features $\times$ $3$ pairs $= 6$ tests, setting the threshold at $\alpha = 0.05 / 6 \approx 0.0083$.

In [17]:
# Pairwise Mann-Whitney for cooler_condition
# Pairs: 3 vs 20, 3 vs 100, 20 vs 100
# Bonferroni correction within the pairwise comparison family:
# 2 features × 3 pairs = 6 tests

cooler_pairs = [(3, 20), (3, 100), (20, 100)]
cooler_features = ['TS1_mean', 'TS1_std']
PAIRWISE_ALPHA = 0.05 / (len(cooler_pairs) * len(cooler_features))  # 0.05 / 6

cooler_pairwise_results = []

for var in cooler_features:
    for a, b in cooler_pairs:
        group_a = hydraulics.loc[hydraulics['cooler_condition'] == a, var].dropna()
        group_b = hydraulics.loc[hydraulics['cooler_condition'] == b, var].dropna()

        stat, p = mannwhitneyu(group_a, group_b, alternative='two-sided')

        n1, n2 = len(group_a), len(group_b)
        rank_biserial = 1 - (2 * stat) / (n1 * n2)

        med_a = group_a.median()
        med_b = group_b.median()
        if med_a > med_b:
            direction = f'{a} > {b}'
        elif med_a < med_b:
            direction = f'{b} > {a}'
        else:
            direction = 'equal'

        cooler_pairwise_results.append({
            'variable':      var,
            'comparison':    f'{a} vs {b}',
            'n_a':           n1,
            'n_b':           n2,
            'median_a':      med_a,
            'median_b':      med_b,
            'direction':     direction,
            'u_stat':        stat,
            'p_value':       p,
            'rank_biserial': rank_biserial,
            'alpha_bonf':    PAIRWISE_ALPHA,
            'significant':   p < PAIRWISE_ALPHA,
        })

cooler_pairwise_df = pd.DataFrame(cooler_pairwise_results)

# Clean display formatting
for col in ['median_a', 'median_b', 'u_stat', 'rank_biserial']:
    cooler_pairwise_df[col] = cooler_pairwise_df[col].round(4)
cooler_pairwise_df['p_value'] = cooler_pairwise_df['p_value'].apply(lambda p: f"{p:.3e}")

cooler_pairwise_df

,variable,comparison,n_a,n_b,median_a,median_b,direction,u_stat,p_value,rank_biserial,alpha_bonf,significant
0,TS1_mean,3 vs 20,732,732,55.6925,44.9063,3 > 20,524929.0,1.318e-221,-0.9593,0.008333,True
1,TS1_mean,3 vs 100,732,741,55.6925,35.8881,3 > 100,541875.0,4.060e-241,-0.9980,0.008333,True
2,TS1_mean,20 vs 100,732,741,44.9063,35.8881,20 > 100,542100.0,1.625e-241,-0.9988,0.008333,True
3,TS1_std,3 vs 20,732,732,0.2279,0.1961,3 > 20,475016.0,1.295e-144,-0.7730,0.008333,True
4,TS1_std,3 vs 100,732,741,0.2279,0.1507,3 > 100,540836.0,2.753e-239,-0.9942,0.008333,True
5,TS1_std,20 vs 100,732,741,0.1961,0.1507,20 > 100,529664.0,4.859e-220,-0.9530,0.008333,True


#### Observations

All six pairwise Mann-Whitney comparisons remain significant under the Bonferroni-corrected threshold ($\alpha = 0.0083$), with p-values ranging from $10^{-144}$ to $10^{-241}$. The direction of every comparison is consistent with the monotonic gradient observed in the omnibus medians - more degraded cooler states show higher `TS1_mean` and `TS1_std` values across every pair of levels.

Rank-biserial effect sizes are very large in absolute magnitude, exceeding $0.95$ for five of the six comparisons. The exception is `TS1_std` between cooler states $3\%$ and $20\%$, where the rank-biserial value of approximately $0.77$ still indicates a large effect, but a meaningfully weaker one than in the other contrasts. This suggests that within-cycle thermal variability separates the two most degraded cooler states less cleanly than mean temperature does, while both features clearly separate degraded states from full efficiency.

These pairwise results confirm that the cooler-condition signal is graded across all three degradation levels, rather than being driven only by an extreme best-versus-worst contrast. Both `TS1_mean` and `TS1_std` qualify as strong discriminators of cooler condition, with `TS1_mean` appearing slightly preferable when distinguishing between the two non-healthy degradation states.

### Pump leakage

The `pump_leakage` label has three levels - $0$ (no leakage), $1$ (weak leakage), and $2$ (severe leakage). The healthy class dominates the cycle distribution ($1221$ cycles versus $492$ each for weak and severe leakage). The formal contrast is the endpoint comparison between full health ($0$) and severe leakage ($2$) using Mann-Whitney U. The weak leakage state ($1$) is omitted from the formal test but may be referenced descriptively where relevant.

In [18]:
# Mann-Whitney U for pump_leakage: endpoint contrast 0 vs 2
# Bonferroni correction uses the global k = 5 family

pump_features = ['EPS1_mean']

pump_results = []

for var in pump_features:
    healthy = hydraulics.loc[hydraulics['pump_leakage'] == 0, var].dropna()
    severe = hydraulics.loc[hydraulics['pump_leakage'] == 2, var].dropna()

    stat, p = mannwhitneyu(healthy, severe, alternative='two-sided')

    n1, n2 = len(healthy), len(severe)
    rank_biserial = 1 - (2 * stat) / (n1 * n2)

    med_h = healthy.median()
    med_s = severe.median()

    if med_h > med_s:
        direction = 'healthy > severe'
    elif med_h < med_s:
        direction = 'severe > healthy'
    else:
        direction = 'equal'

    pump_results.append({
        'variable': var,
        'comparison': '0 (healthy) vs 2 (severe leakage)',
        'n_healthy': n1,
        'n_severe': n2,
        'median_healthy': med_h,
        'median_severe': med_s,
        'direction': direction,
        'u_stat': stat,
        'p_value': p,
        'rank_biserial': rank_biserial,
        'alpha_bonf': GLOBAL_ALPHA,
        'significant': p < GLOBAL_ALPHA,
    })

pump_df = pd.DataFrame(pump_results)

# Clean display formatting
pump_df['median_healthy'] = pump_df['median_healthy'].round(4)
pump_df['median_severe'] = pump_df['median_severe'].round(4)
pump_df['u_stat'] = pump_df['u_stat'].round(4)
pump_df['rank_biserial'] = pump_df['rank_biserial'].round(4)
pump_df['p_value'] = pump_df['p_value'].apply(lambda p: f"{p:.3e}")

pump_df

,variable,comparison,n_healthy,n_severe,median_healthy,median_severe,direction,u_stat,p_value,rank_biserial,alpha_bonf,significant
0,EPS1_mean,0 (healthy) vs 2 (severe leakage),1221,492,2455.9669,2557.5761,severe > healthy,115092.0,5.366e-89,0.6168,0.01,True


#### Observations

The Mann-Whitney U test produces a highly significant result for `EPS1_mean` between the healthy and severely-leaking pump states ($U = 115092$, $p \approx 5 \times 10^{-89}$, far below the corrected threshold $\alpha = 0.010$). This provides strong support for the hydraulic part of Hypothesis 1 in the case of `pump_leakage` at the endpoint contrast.

The direction of the effect is physically interpretable. Median motor-power-related load is approximately 2557.6 in the severe-leakage group and 2456.0 in the healthy group, indicating that a leaking pump is associated with higher sustained motor load. This is consistent with the expectation that degraded pump performance increases the power demand needed to maintain operation.

The rank-biserial effect size of 0.62 lies in the large range, but is meaningfully smaller than the effect sizes obtained for `cooler_condition` (above 0.95 for most pairwise comparisons). This indicates that while the pump-leakage signal is statistically robust, the separation between healthy and severely leaking pumps is less clean than the separation observed for cooler-condition states. The weak-leakage state (`1`) is not formally tested here, but would be expected to lie between these two endpoints.

### Stable vs unstable cycles

The `stable_flag` label is a binary indicator distinguishing stable cycles ($0$) from unstable cycles ($1$), where the unstable category corresponds to cycles in which static operating conditions had not been fully reached. Two features are tested formally: `TS1_std` and `VS1_std`, representing temperature variability and vibration variability, respectively.

In [19]:
# Mann-Whitney U for stable_flag: binary contrast 0 (stable) vs 1 (unstable)
# Bonferroni correction uses the global k = 5 family

stable_features = ['TS1_std', 'VS1_std']

stable_results = []

for var in stable_features:
    stable = hydraulics.loc[hydraulics['stable_flag'] == 0, var].dropna()
    unstable = hydraulics.loc[hydraulics['stable_flag'] == 1, var].dropna()

    stat, p = mannwhitneyu(stable, unstable, alternative='two-sided')

    n1, n2 = len(stable), len(unstable)
    rank_biserial = 1 - (2 * stat) / (n1 * n2)

    med_s = stable.median()
    med_u = unstable.median()

    if med_s > med_u:
        direction = 'stable > unstable'
    elif med_s < med_u:
        direction = 'unstable > stable'
    else:
        direction = 'equal'

    stable_results.append({
        'variable': var,
        'comparison': '0 (stable) vs 1 (unstable)',
        'n_stable': n1,
        'n_unstable': n2,
        'median_stable': med_s,
        'median_unstable': med_u,
        'direction': direction,
        'u_stat': stat,
        'p_value': p,
        'rank_biserial': rank_biserial,
        'alpha_bonf': GLOBAL_ALPHA,
        'significant': p < GLOBAL_ALPHA,
    })

stable_df = pd.DataFrame(stable_results)

stable_df_display = stable_df.copy()


stable_df_display['median_stable'] = stable_df_display['median_stable'].round(4)
stable_df_display['median_unstable'] = stable_df_display['median_unstable'].round(4)
stable_df_display['u_stat'] = stable_df_display['u_stat'].round(4)
stable_df_display['rank_biserial'] = stable_df_display['rank_biserial'].round(4)
stable_df_display['p_value'] = stable_df_display['p_value'].apply(lambda p: f"{p:.3e}")

stable_df_display[
    ['variable', 'comparison', 'median_stable', 'median_unstable',
     'direction', 'p_value', 'rank_biserial', 'significant']
]

,variable,comparison,median_stable,median_unstable,direction,p_value,rank_biserial,significant
0,TS1_std,0 (stable) vs 1 (unstable),0.1922,0.2002,unstable > stable,5.584e-13,0.1868,True
1,VS1_std,0 (stable) vs 1 (unstable),0.0316,0.0367,unstable > stable,1.362e-04,0.0988,True


#### Observations

The Mann-Whitney U test produces significant results for both `TS1_std` and `VS1_std` between stable and unstable cycles (p-values $5.6 \times 10^{-13}$ and $1.4 \times 10^{-4}$, both below $\alpha = 0.010$). These results support the hydraulic part of Hypothesis 1 in the case of `stable_flag`.

The direction of effect is consistent for both features: unstable cycles show higher within-cycle variability than stable cycles, in line with the physical interpretation that unstable cycles correspond to transient periods in which static operating conditions have not been fully reached.

The rank-biserial effect sizes are notably small, however: $0.19$ for `TS1_std` and $0.10$ for `VS1_std`. Compared with the very large effects observed for `cooler_condition` (above $0.95$) and the moderate-to-large effect for `pump_leakage` ($0.62$), the `stable_flag` distinction produces only modest separation. The result is statistically robust because of the large sample sizes, but the operational separation between stable and unstable cycles is weak; most unstable cycles still overlap heavily with the stable-cycle distribution. This is consistent with the descriptive Notebook 02 finding that temperature and vibration variability are only weak-to-moderate indicators of
cycle stability.

### Hydraulic interpretation

Three condition labels are tested formally, with results that vary substantially in strength.

`cooler_condition` produces the strongest formal evidence in the entire project. Both `TS1_mean` and `TS1_std` separate the three cooler levels overwhelmingly (Kruskal-Wallis $p < 10^{-300}$), with rank-biserial effect sizes above 0.95 in five of the six pairwise comparisons. The signal is graded across all three degradation states and is physically interpretable as the thermal consequence of reduced cooler efficiency.

`pump_leakage` produces a robust but weaker result. `EPS1_mean` distinguishes healthy from severe leakage with $p \approx 5 \times 10^{-89}$ and a large effect size of 0.62. The direction is consistent with the physical expectation that degraded pump performance is associated with higher motor load. The middle leakage state is not formally tested.

`stable_flag` produces statistically significant but operationally weak results. Both `TS1_std` and `VS1_std` distinguish stable from unstable cycles ($p < 10^{-3}$), with the predicted direction, but rank-biserial effect sizes of 0.19 and 0.10 indicate that individual unstable cycles overlap heavily with the stable distribution.

Taken together, the three labels span a clear gradient of signal strength: the cooler-condition signal is dominant, the pump-leakage signal is robust, and the stable-flag signal is detectable but weak. All three results support Hypothesis 1 for the hydraulic dataset, though the operational meaningfulness of that support varies considerably across labels.

---

## Reliability metrics

Two reliability-oriented measures are calculated as supporting context for Hypothesis 3: Mean Time To Repair (MTTR) and Mean Time Between Failures (MTBF). For MetroPT-3, both metrics are computed directly from the four  documented failure events. For the hydraulic dataset, neither metric translates cleanly because the data was produced under controlled laboratory conditions rather than as a continuous operational record - what is reported instead is the proportion of cycles in each labeled degradation state.

### MetroPT-3 reliability

MetroPT-3 includes four documented air-leak failure events with start and end timestamps. These allow rough estimation of repair duration and time between failures. Because the number of events is small, the resulting metrics should be interpreted as descriptive context rather than as stable engineering reliability estimates.

In [20]:
metro_failures_sorted = metro_failures.sort_values('start_time').reset_index(drop=True)

# MTTR: mean of failure durations
metro_mttr = metro_failures_sorted['duration'].mean()

# MTBF: gaps between consecutive failures (end of failure i -> start of failure i+1)
gaps = (
    metro_failures_sorted['start_time'].iloc[1:].values -
    metro_failures_sorted['end_time'].iloc[:-1].values
)
metro_mtbf = pd.Series(pd.to_timedelta(gaps)).mean()

# Per-failure detail for the report
metro_failures_sorted['gap_to_next'] = pd.Series(
    [pd.NaT] * len(metro_failures_sorted),
    dtype='timedelta64[ns]'
)
metro_failures_sorted.loc[:len(metro_failures_sorted) - 2, 'gap_to_next'] = pd.to_timedelta(gaps)

print(f"MetroPT-3 reliability metrics (n = {len(metro_failures_sorted)} failures)\n")
print(f"  MTTR (Mean Time To Repair):    {metro_mttr}")
print(f"  MTBF (Mean Time Between Failures): {metro_mtbf}\n")

print("Per-failure detail:")
display_cols = ['failure_id', 'start_time', 'end_time', 'duration', 'gap_to_next']
print(metro_failures_sorted[display_cols].to_string(index=False))

MetroPT-3 reliability metrics (n = 4 failures)

  MTTR (Mean Time To Repair):    0 days 21:52:15
  MTBF (Mean Time Between Failures): 28 days 09:10:20

Per-failure detail:
 failure_id          start_time            end_time        duration      gap_to_next
          1 2020-04-18 00:00:00 2020-04-18 23:59:00 0 days 23:59:00 40 days 23:31:00
          2 2020-05-29 23:30:00 2020-05-30 06:00:00 0 days 06:30:00  6 days 04:00:00
          3 2020-06-05 10:00:00 2020-06-07 14:30:00 2 days 04:30:00 38 days 00:00:00
          4 2020-07-15 14:30:00 2020-07-15 19:00:00 0 days 04:30:00              NaT


#### Observations

Using the documented MetroPT-3 failure intervals, the average repair duration is approximately $21$ hours and $25$ minutes (MTTR), while the average time between consecutive failures is approximately $28$ days and $9$ hours (MTBF). These values should be interpreted descriptively because they are based on only four reported incidents, but they still provide useful operational context.

The MTTR estimate shows that the documented air-leak failures were not instantaneous anomalies and they corresponded to extended interruption or repair periods. The MTBF estimate indicates that failures were relatively infrequent across the seven-month monitoring period, occurring roughly every four weeks on average.

Within this project, these reliability metrics do not serve as primary evidence on their own. Instead, they support the broader interpretation that the monitored compressor experienced intermittent but operationally meaningful failure episodes, which is consistent with the sensor-based analysis of pre-failure behavior.

### Hydraulic context

The hydraulic dataset is cycle-based rather than a continuous operational record with timestamped failures and repairs. For that reason, standard MTBF and MTTR definitions do not apply directly. Instead, the proportion of cycles observed in each relevant degradation or stability state is reported. These values provide condition context, but should not be interpreted as time-based reliability metrics comparable to those calculated for MetroPT-3.

In [21]:
# Compact reliability-context summary for the hydraulic dataset

hydraulic_state_cols = [
    'cooler_condition',
    'pump_leakage',
    'stable_flag'
]

state_records = []

for col in hydraulic_state_cols:
    counts = hydraulics[col].value_counts().sort_index()
    shares = hydraulics[col].value_counts(normalize=True).sort_index()

    for level in counts.index:
        state_records.append({
            'label': col,
            'level': level,
            'n_cycles': counts[level],
            'share': round(shares[level], 4)
        })

hydraulic_state_summary = pd.DataFrame(state_records)

hydraulic_state_summary

,label,level,n_cycles,share
0,cooler_condition,3,732,0.3320
1,cooler_condition,20,732,0.3320
2,cooler_condition,100,741,0.3361
3,pump_leakage,0,1221,0.5537
4,pump_leakage,1,492,0.2231
5,pump_leakage,2,492,0.2231
6,stable_flag,0,1449,0.6571
7,stable_flag,1,756,0.3429


#### Observations

The hydraulic dataset provides substantial representation of both healthy and degraded operating states, but these proportions should be interpreted as condition context rather than as time-based reliability metrics.

`cooler_condition` is almost perfectly balanced across its three levels, which supports fair comparison of thermal degradation states. `pump_leakage` is more concentrated in the healthy class, but both weak and severe leakage remain well represented. `stable_flag` shows that about one-third of all cycles are unstable, confirming that the dataset contains a meaningful share of non-steady operating behavior.

These proportions help explain why the hydraulic dataset is well suited to condition-discrimination testing, but they are not directly comparable to MTBF or MTTR because the data does not describe a continuous operational timeline.

### Reliability interpretation

Hypothesis 3 is supported mainly through the MetroPT-3 reliability context. The MTTR and MTBF estimates confirm that the monitored compressor experienced intermittent but operationally meaningful failure episodes: the failures were not instantaneous and recurred at roughly four-week intervals across the monitoring period. This is consistent with the sensor-based evidence that pre-failure conditions produce detectable shifts in thermal and pressure-related variables.

The hydraulic dataset contributes indirect context only. The condition-state proportions confirm that the dataset contains substantial representation of both healthy and degraded operating states, which supports the validity of the condition-based comparisons conducted in the previous section. However, because the data doesn't describe a continuous operational timeline, no direct MTBF or MTTR analogue is available for the hydraulic system.

---

## Cross-dataset interpretation

The two datasets differ fundamentally in structure and labeling, and are therefore analyzed separately in the preceding sections. This section synthesizes the findings across both systems in relation to the three shared physical sensing concepts: temperature, pressure, and electrical or load-related behavior, and evaluates the degree of cross-dataset concordance.

### Temperature

Temperature is the strongest cross-dataset signal. In MetroPT-3, `Oil_temperature` shows a consistent pre-failure directional shift at the 6-hour window across all three retained failure events, providing exploratory-supportive evidence despite limited formal power. In the hydraulic dataset, `TS1_mean` and `TS1_std` produce the strongest formal findings in the entire project, with near-perfect separation across all three `cooler_condition` levels. The convergence of evidence across two structurally different systems supports the interpretation that thermal behavior is a meaningful indicator of condition change in fluid-power systems.

### Pressure

Pressure is only partially supported as a cross-dataset signal. In MetroPT-3, `TP2` shows a partial pre-failure directional pattern: two of three retained failures exhibit lower pre-failure medians, but the third is flat, placing it as a weaker secondary candidate. In the hydraulic dataset, pressure-related features play a secondary role rather than acting as dominant discriminators.

The partial concordance suggests that pressure-related sensing concepts respond to degradation in both systems, but the signal is less consistent and less separable than the thermal signal. Pressure is therefore treated as supporting rather than primary cross-dataset evidence.

### Electrical and load-related behavior

Electrical and load-related behavior is the most divergent concept across the two datasets. In MetroPT-3, `Motor_current` is excluded from formal testing because it shows mixed directional patterns across failure events at every tested window length, indicating that its variation reflects operating-regime changes rather than a pre-failure signature. In the hydraulic dataset, `EPS1_mean` provides a robust signal for `pump_leakage`, distinguishing healthy from severely leaking pumps with a large effect size of $0.62$.

This divergence is physically interpretable - the two systems monitor different components under different operational contexts. In MetroPT-3, motor current reflects compressor load variability across operating regimes. In the hydraulic dataset, motor power is measured per cycle under controlled conditions, making it a cleaner discriminator of pump degradation state. The divergence therefore reflects a structural difference in how the concept is captured rather than a contradiction in the underlying physics. This difference is also partly measurement-based: MetroPT-3 records motor current, whereas the hydraulic dataset uses motor-power-related features.

### Cross-dataset conclusion

Hypothesis 2 is partially supported. Concordance is strongest for temperature, where both datasets show degradation-related thermal shifts that are physically interpretable, even though the direction of the shift is dataset-specific. Pressure shows partial concordance, with a weaker and less consistent signal in both systems. Electrical and load-related behavior is the most divergent concept: robust in the hydraulic dataset but absent as a stable pre-failure signal in MetroPT-3.

This partial support reflects the structural asymmetry between the two datasets rather than a fundamental disagreement in the underlying physics. MetroPT-3 captures broad sensing concepts under real field operation, where operating-regime variability can mask degradation signals. The hydraulic dataset captures the same general concepts under controlled cycle-based conditions, where degradation states are explicitly labeled and feature separation is stronger. Temperature therefore emerges as the most transferable concept across the two monitoring contexts.

---

## Conclusions

This notebook evaluates the three project hypotheses through formal statistical testing and reliability context. The results are summarized
hypothesis by hypothesis below.

### Hypothesis 1

Hypothesis 1 is supported, but with substantially different strength across the two datasets. In the hydraulic dataset, all three formally tested condition labels produce significant results, with `cooler_condition` showing the strongest evidence in the entire project. The signal is graded across degradation levels and physically interpretable. In MetroPT-3, the formal tests do not reach significance due to the small number of matched failure events available after the coverage filter. The directional evidence remains meaningful nonetheless: `Oil_temperature` shows the clearest and most consistent pre-failure pattern, while `H1` provides weaker supporting evidence.

### Hypothesis 2

Hypothesis 2 is partially supported. Temperature is the strongest shared concept, showing condition-related thermal shifts in both systems. Pressure shows partial concordance. Electrical and load-related behavior is divergent between the two datasets, which is attributable to structural differences in how the concept is captured rather than to a contradiction in the underlying physics.

### Hypothesis 3

Hypothesis 3 is supported mainly through the MetroPT-3 reliability context. The MTTR and MTBF estimates confirm that the monitored compressor experienced intermittent but operationally meaningful failure episodes, recurring at roughly four-week intervals across the monitoring period. This is consistent with the sensor-based evidence that pre-failure conditions produce detectable shifts in thermal and pressure-related variables. The hydraulic dataset contributes indirect context through condition-state proportions, but does not provide directly comparable time-based reliability metrics.

### Final closing note

Fluid-power degradation signals exist in both datasets, but their detectability depends strongly on data structure and monitoring context. The hydraulic dataset, with its controlled cycle-based design and explicit condition labels, produces clean and strong separation across degradation states. MetroPT-3, as a continuous field-monitoring record with sparse failure annotations, produces weaker formal evidence but captures the kind of operational variability that real predictive maintenance systems must contend with. Together, the two datasets illustrate both the potential and the limitations of sensor-based degradation detection in fluid-power systems.

---